# ACE-Step 1.5 — Gradio UI + LoRA/LoKr training on Google Colab

This notebook installs ACE-Step 1.5, persists models / datasets / training outputs on
Google Drive, and launches the full Gradio interface (generation **and** the LoRA / LoKr
training tabs) behind a public `gradio.live` link.

**Before you start:** `Runtime → Change runtime type → GPU` (A100 or L4 recommended for
training; a T4 works for generation and small LoRAs with gradient checkpointing).

Run the cells from top to bottom. The last cell keeps running as long as the UI is up.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Settings

- `USE_DRIVE`: keep checkpoints, datasets and LoRA outputs on Google Drive so they survive
  the end of the Colab session (strongly recommended for training).
- `UI_PASSWORD`: the `gradio.live` link is public — set a password to protect it.

In [ ]:
#@title Settings { display-mode: "form" }
REPO_URL = "https://github.com/walidb27/ACE-Step-1.5"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/ACE-Step"  #@param {type:"string"}
UI_USERNAME = "admin"  #@param {type:"string"}
UI_PASSWORD = ""  #@param {type:"string"}
UI_LANGUAGE = "en"  #@param ["en", "zh", "ja", "pt", "he"]
PREDOWNLOAD_MODELS = True  #@param {type:"boolean"}

REPO_DIR = "/content/ACE-Step-1.5"

## 3. Mount Google Drive (optional)

In [ ]:
import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print("Persistent storage:", DRIVE_DIR)
else:
    print("Drive disabled: everything will be lost when the session ends.")

## 4. Install ACE-Step

Clones the repository and installs the locked dependencies with `uv` in a Python 3.11
virtual environment (matches the prebuilt `flash-attn` wheel). Takes ~5 minutes.

In [ ]:
import os

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]

# Colab sets MPLBACKEND to its inline backend, which the venv's matplotlib rejects.
os.environ["MPLBACKEND"] = "Agg"

%cd {REPO_DIR}
!uv sync --python 3.11

## 5. Persist folders on Google Drive

- `checkpoints` and `gradio_outputs` are symlinked to Drive.
- `datasets`, `lora_output` and `lokr_output` stay **real local folders**: the training
  code rejects paths that resolve outside the repository, so symlinks to Drive would make
  Scan / Preprocess / Train fail with "Rejected unsafe directory path". They are copied
  from Drive now, and a background job copies them back to Drive every 2 minutes.

Put your audio datasets in `<DRIVE_DIR>/datasets/<name>` and use `./datasets/<name>`
as the path in the UI. Re-run this cell after adding files to Drive.

In [ ]:
import os
import shutil
import subprocess

LINKED_DIRS = ["checkpoints", "gradio_outputs"]
SYNCED_DIRS = ["datasets", "lora_output", "lokr_output"]
SYNC_INTERVAL_S = 120

def link_to_drive(name: str) -> None:
    local = os.path.join(REPO_DIR, name)
    remote = os.path.join(DRIVE_DIR, name)
    os.makedirs(remote, exist_ok=True)
    if os.path.islink(local):
        return
    if os.path.isdir(local):
        shutil.copytree(local, remote, dirs_exist_ok=True)
        shutil.rmtree(local)
    os.symlink(remote, local)

def pull_from_drive(name: str) -> None:
    local = os.path.join(REPO_DIR, name)
    remote = os.path.join(DRIVE_DIR, name)
    os.makedirs(remote, exist_ok=True)
    if os.path.islink(local):
        os.unlink(local)  # replace a symlink left by an older notebook version
    os.makedirs(local, exist_ok=True)
    subprocess.run(["rsync", "-a", remote + "/", local + "/"], check=True)

def start_background_sync() -> None:
    script = os.path.join(os.path.dirname(REPO_DIR), "acestep_drive_sync.sh")
    pushes = "\n".join(
        f"  rsync -a '{REPO_DIR}/{n}/' '{DRIVE_DIR}/{n}/'" for n in SYNCED_DIRS
    )
    with open(script, "w") as f:
        f.write(f"while true; do\n  sleep {SYNC_INTERVAL_S}\n{pushes}\ndone\n")
    subprocess.run(["pkill", "-f", script], check=False)
    subprocess.Popen(["bash", script], start_new_session=True)

if USE_DRIVE:
    if shutil.which("rsync") is None:
        !apt-get -qq install -y rsync
    for name in LINKED_DIRS:
        link_to_drive(name)
        print(f"./{name} -> {os.path.join(DRIVE_DIR, name)} (symlink)")
    for name in SYNCED_DIRS:
        pull_from_drive(name)
        print(f"./{name} <- copied from {os.path.join(DRIVE_DIR, name)}")
    start_background_sync()
    print(f"Background sync to Drive every {SYNC_INTERVAL_S}s started.")
else:
    for name in LINKED_DIRS + SYNCED_DIRS:
        os.makedirs(os.path.join(REPO_DIR, name), exist_ok=True)

## 6. Download the models (optional)

Downloads the default models into `./checkpoints` (on Drive if enabled), so later sessions
start immediately. The UI can also download them on demand.

In [ ]:
import os

os.environ["MPLBACKEND"] = "Agg"
%cd {REPO_DIR}
if PREDOWNLOAD_MODELS:
    !uv run acestep-download

## 7. Launch the Gradio UI

Wait for the line `Running on public URL: https://xxxx.gradio.live` and open that link.
In the UI:

1. **Service configuration**: pick the models and click *Initialize Service*.
2. **LoRA Training** tab: dataset path `./datasets/<your_dataset>`, then Scan →
   Auto-label → Save → Preprocess → Train. Outputs go to `./lora_output` (Drive).

See `docs/en/LoRA_Training_Tutorial.md` for the full training workflow.
Stop the cell (■) to shut the UI down.

In [ ]:
import os
import shlex

os.environ["MPLBACKEND"] = "Agg"
%cd {REPO_DIR}
auth = ""
if UI_PASSWORD:
    auth = (
        f"--auth-username {shlex.quote(UI_USERNAME)} --auth-password {shlex.quote(UI_PASSWORD)}"
    )
else:
    print("WARNING: no password set, anyone with the link can use your GPU.")

!uv run acestep --share --server-name 0.0.0.0 --port 7860 --language {UI_LANGUAGE} {auth}

## Tips

- **Disconnections**: Colab stops idle sessions. Keep the tab open; with Drive enabled,
  checkpoints saved every *N* epochs survive and can be resumed from
  `./lora_output/checkpoints/epoch_XXX`.
- **Low VRAM (T4, 16 GB)**: enable *gradient checkpointing* and use batch size 1.
- **Download a LoRA**: it is copied to Drive in `MyDrive/ACE-Step/lora_output/` (or
  `lokr_output/`) within 2 minutes of being written. Wait a few minutes after training
  before stopping the runtime.